In [24]:
#imports
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV

import joblib
from sksurv.util import Surv
from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import concordance_index_censored,  as_concordance_index_ipcw_scorer

In [11]:
# loading data
df = pd.read_csv("./data/framingham.csv")

event_col = "CVD"
time_col = "TIMECVD"

features = ["SEX", 
            "AGE", 
            "SYSBP", 
            "DIABP", 
            "BPMEDS", 
            "CURSMOKE", 
            "CIGPDAY", 
            "TOTCHOL", 
            "BMI", 
            "GLUCOSE", 
            "DIABETES",
            "HEARTRTE",
            ]

data = df[features + [event_col, time_col]].copy()
data = data.dropna(subset=[event_col, time_col])

In [12]:
X = data[features]
y = Surv.from_dataframe(
    event = event_col,
    time = time_col,
    data = data
)

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=2026
)

In [25]:
numeric_features = features
preprocessor = ColumnTransformer(
    transformers=[
        ("num", 
         Pipeline([
             ("imputer", SimpleImputer(strategy="median")),
             ("scaler", StandardScaler())
         ]),
         numeric_features
         )
    ]
)

rsf = RandomSurvivalForest(
    random_state=2026,
    n_jobs=-1,
    n_estimators=100,
    min_samples_split=5,
    min_samples_leaf=10,
    max_features='sqrt'
)

model = Pipeline([
    ("preprocess", preprocessor),
    ("rsf", rsf)
])

scored_model = as_concordance_index_ipcw_scorer(model)

param_grid = {
    "estimator__rsf__n_estimators": [100],
    "estimator__rsf__min_samples_split": [5],
    "estimator__rsf__min_samples_leaf": [10],
    "estimator__rsf__max_features": ["sqrt"],
}


grid_search = GridSearchCV(
    estimator=scored_model,
    param_grid=param_grid,
    cv=5,
    n_jobs=-1,
)

#grid_search.fit(X_train, y_train)
#print(grid_search.best_params_)
#print(grid_search.best_score_)

model.fit(X_train, y_train)


joblib.dump(model, 'model.pkl')

['model.pkl']

In [21]:

risk_scores = model.predict(X_test)
c_index = concordance_index_censored(
    y_test["CVD"],
    y_test["TIMECVD"],
    risk_scores
)[0]

print("Concordance index:", c_index)

Concordance index: 0.7301154978724076


In [22]:
surv_funcs = model.predict_survival_function(X_test.iloc[:5])

for i, surv_fn in enumerate(surv_funcs):
    print(f"Patient {i}")
    print("Survival probability at 5 years:", surv_fn(365 * 5))
    print("Survival probability at 10 years:", surv_fn(365 * 10))

Patient 0
Survival probability at 5 years: 0.6629336229556333
Survival probability at 10 years: 0.526327258005841
Patient 1
Survival probability at 5 years: 0.9327780004111312
Survival probability at 10 years: 0.8796594774711854
Patient 2
Survival probability at 5 years: 0.920193839271231
Survival probability at 10 years: 0.859532109312716
Patient 3
Survival probability at 5 years: 0.9969057337209226
Survival probability at 10 years: 0.9965532081977387
Patient 4
Survival probability at 5 years: 0.9871131181116687
Survival probability at 10 years: 0.9687204504239318
